# Recognition and extraction are separate measurements

Demonstration documents only. These notebooks inspect the committed evidence; they do not claim a separate full-dataset training run.

In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
e = json.loads((root / 'results/manifest.json').read_text(encoding='utf-8'))
print('Population:', e['scope'])


Population: {'generated': {'train': 48, 'validation': 24, 'test': 24}, 'cord': {'train': 800, 'validation': 100, 'test': 100}, 'limitations': ['Full original CORD 800/100/100 splits are used, with a project-specific subtotal/tax/total and line-item mapping, not the official benchmark metric.', 'Generated invoices share four templates; performance does not establish generalization to unseen vendors or layouts.', 'Small test populations give wide intervals; the chosen costs are illustrative (review 1, wrong approval 20).', 'CORD gold words cover annotated semantic lines; full-page OCR CER/WER includes unannotated text.', 'Gold/real OCR input gaps include token segmentation and layout conventions, so they are not pure additive character-recognition loss.', 'Rules are conservative on receipts without invoice date, vendor and PO evidence, so those receipts go to review.', 'Other-class classification uses generated text tokens; no real miscellaneous mailroom documents were evaluated.', 'Extr

In [2]:
rows = pd.DataFrame([{'source':r['source'], 'extractor':r['backend'], 'input':r['ocr_engine'],
                     'documents':r['n'], 'field_f1':r['field_f1']['value']} for r in e['extraction']])
print(rows.to_string(index=False))
print(pd.DataFrame(e['ocr']).to_string(index=False))
assert rows['input'].nunique() >= 2, 'Gold words do not establish OCR quality'

   source extractor     input  documents  field_f1
generated     rules      gold         24  1.000000
generated     rules tesseract         24  1.000000
generated    tokens      gold         24  1.000000
generated    tokens tesseract         24  1.000000
     cord     rules      gold        100  0.635069
     cord     rules tesseract        100  0.248264
     cord    tokens      gold        100  0.854514
     cord    tokens tesseract        100  0.310069
   source    engine   n split version                                                              preprocessing                                                                                                                                                                  cer                                                                                                                                                                  wer
generated tesseract  24  test   5.3.0 EXIF orientation; RGB; upscale width below 900px; no thresho

## Rejected shortcut / negative control

In [3]:
# Rejected shortcut: a gold-only result cannot measure recognition errors.
gold_only = rows[rows['input'].str.contains('gold')]
print('Gold-only rows:', len(gold_only))
print('Real OCR rows removed by this shortcut:', len(rows) - len(gold_only))
assert len(gold_only) < len(rows)

Gold-only rows: 4
Real OCR rows removed by this shortcut: 4
